# Type-B Training Pipeline

Supports two modes — set `MODE` in the Config cell:
- `'local'` : runs against local repo (VS Code + local kernel)
- `'colab'` : runs on Google Colab (mounts Drive, clones GitHub repo)

## Cell 1 — Config

In [ ]:
# ── MODE ──────────────────────────────────────────────────────────────────────
MODE = 'colab'   # 'local' or 'colab'

# ── Local config (used when MODE='local') ─────────────────────────────────────
LOCAL_REPO_DIR = '/Users/yeon/work/UoB_CourseWork/uob-ds-intro-to-ai-final-cw-2026'

# ── Colab config (used when MODE='colab') ─────────────────────────────────────
# Token is loaded from Colab Secrets (left sidebar → key icon → add GITHUB_TOKEN)
# Never hardcode the token here — GitHub will auto-revoke it.
GITHUB_REPO_URL = 'https://github.com/Rylie-KIM/uob-ds-intro-to-ai-final-cw-2026'
DRIVE_BASE      = '/content/drive/MyDrive/uob-ds-ai'

# ── Experiment config (both modes) ────────────────────────────────────────────
MODELS     = ['cnn_1layer']      # ['cnn_1layer', 'cnn_2layer', 'cnn', 'alexnet']
EMBEDDINGS = ['tfidf_lsa']       # ['tfidf_lsa', 'sbert', 'bert_mean', ...]
EPOCHS     = 30
SKIP_EMBED = False
RUN_GEMINI = False
# ──────────────────────────────────────────────────────────────────────────────

print(f'MODE = {MODE}')

## Cell 2 — Setup (auto-selects local or Colab path)

In [ ]:
import os, sys, subprocess, shutil, time
from pathlib import Path

if MODE == 'local':
    REPO_DIR = LOCAL_REPO_DIR
    print(f'Local mode. Repo: {REPO_DIR}')

elif MODE == 'colab':
    from google.colab import drive, userdata
    drive.mount('/content/drive')

    REPO_DIR      = '/content/repo'
    DRIVE_DATA    = f'{DRIVE_BASE}/data'
    DRIVE_RESULTS = f'{DRIVE_BASE}/results'

    # Build authenticated clone URL from Colab Secrets
    try:
        token = userdata.get('GITHUB_TOKEN')
        repo_url_auth = GITHUB_REPO_URL.replace('https://', f'https://{token}@')
    except Exception:
        print('WARNING: GITHUB_TOKEN not found in Colab Secrets.')
        repo_url_auth = GITHUB_REPO_URL

    # Clone or pull repo
    if not os.path.exists(REPO_DIR):
        print('Cloning repo...')
        subprocess.run(['git', 'clone', repo_url_auth, REPO_DIR], check=True)
    else:
        print('Repo exists. Pulling latest...')
        subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

    # ── Copy images to Colab local disk ───────────────────────────────────────
    LOCAL_IMG_BASE = Path('/content/images')
    for dtype in ['type-b']:
        local_img = LOCAL_IMG_BASE / dtype
        drive_img = Path(DRIVE_DATA) / 'images' / dtype
        repo_img  = Path(REPO_DIR) / 'src' / 'data' / 'images' / dtype

        if local_img.exists() and any(local_img.iterdir()):
            n = len(list(local_img.glob('*.png')))
            print(f'[skip] images/{dtype} already on local disk ({n} files)')
        else:
            local_img.mkdir(parents=True, exist_ok=True)
            src_files = sorted(drive_img.glob('*.png'))
            total     = len(src_files)
            print(f'Copying {total} images ({dtype}) from Drive → /content/images/{dtype}/')
            t0 = time.time()
            for i, src in enumerate(src_files, 1):
                shutil.copy2(str(src), str(local_img / src.name))
                if i % 500 == 0 or i == total:
                    elapsed = time.time() - t0
                    eta     = elapsed / i * (total - i)
                    pct     = i / total * 100
                    print(f'  {i}/{total} ({pct:.0f}%)  elapsed={elapsed:.0f}s  eta={eta:.0f}s', end='\r')
            print(f'\n[done] {total} images copied in {time.time()-t0:.1f}s')

        # Symlink repo → local disk
        if repo_img.is_symlink():
            os.unlink(str(repo_img))
        elif repo_img.exists():
            shutil.rmtree(str(repo_img))
        os.symlink(str(local_img), str(repo_img))
        print(f'[symlinked] repo/src/data/images/{dtype} → /content/images/{dtype}')

    # ── Symlink results → Drive ────────────────────────────────────────────────
    for sub in ['checkpoints', 'metrics', 'figures']:
        drive_sub = Path(DRIVE_RESULTS) / sub
        drive_sub.mkdir(parents=True, exist_ok=True)
        repo_sub = Path(REPO_DIR) / 'src' / 'pipelines' / 'results' / sub
        repo_sub.parent.mkdir(parents=True, exist_ok=True)
        if not repo_sub.exists():
            os.symlink(str(drive_sub), str(repo_sub))
            print(f'[symlinked] results/{sub} → Drive')

    # ── Symlink embedding .pt → Drive ─────────────────────────────────────────
    drive_emb = Path(DRIVE_RESULTS) / 'embeddings'
    drive_emb.mkdir(parents=True, exist_ok=True)
    repo_emb = Path(REPO_DIR) / 'src' / 'embeddings' / 'computed-embeddings' / 'type-b' / 'results'
    repo_emb.mkdir(parents=True, exist_ok=True)
    for pt in drive_emb.glob('*.pt'):
        dest = repo_emb / pt.name
        if not dest.exists():
            os.symlink(str(pt), str(dest))
            print(f'[symlinked] {pt.name}')

    print(f'\nColab mode ready. Repo: {REPO_DIR}')

# Add repo to sys.path
for p in [REPO_DIR, str(Path(REPO_DIR) / 'src'), str(Path(REPO_DIR) / 'src' / 'embeddings' / 'non-pretrained')]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

## Cell 3 — Install Dependencies

In [ ]:
if MODE == 'colab':
    !pip install -q sentence-transformers gensim python-docx datasets
    print('Dependencies installed.')
else:
    print('Local mode — using existing venv. Skipping pip install.')

## Cell 4 — Verify Paths

In [ ]:
checks = {
    'sentences_b.csv' : Path(REPO_DIR) / 'src' / 'data' / 'type-b' / 'sentences_b.csv',
    'image_map_b.csv' : Path(REPO_DIR) / 'src' / 'data' / 'type-b' / 'image_map_b.csv',
    'images/type-b/'  : Path(REPO_DIR) / 'src' / 'data' / 'images' / 'type-b',
    'splits CSV'      : Path(REPO_DIR) / 'src' / 'data' / 'type-b' / 'splits',
    'train_type_b.py' : Path(REPO_DIR) / 'src' / 'pipelines' / 'training' / 'type-b' / 'train_type_b.py',
    'generate_embed'  : Path(REPO_DIR) / 'src' / 'embeddings' / 'computed-embeddings' / 'type-b' / 'generate_embeddings_type_b.py',
}
all_ok = True
for name, path in checks.items():
    ok = path.exists()
    print(f'  {"OK" if ok else "MISSING"}  {name:25s}  {path}')
    if not ok:
        all_ok = False
print()
print('All paths OK — ready to run.' if all_ok else 'Fix missing paths before continuing.')

## Cell 5 — Generate Embeddings

In [ ]:
generate_script  = Path(REPO_DIR) / 'src' / 'embeddings' / 'computed-embeddings' / 'type-b' / 'generate_embeddings_type_b.py'
embed_results_dir = Path(REPO_DIR) / 'src' / 'embeddings' / 'computed-embeddings' / 'type-b' / 'results'

if SKIP_EMBED:
    print('Skipping embedding generation (SKIP_EMBED=True)')
else:
    for emb in EMBEDDINGS:
        pt_path = embed_results_dir / f'{emb}_embedding_result_typeb.pt'
        if pt_path.exists():
            print(f'[skip] {emb} — already exists')
            continue
        print(f'[generating] {emb}...')
        subprocess.run(
            [sys.executable, str(generate_script), '--embedding', emb],
            cwd=REPO_DIR,
            check=True,
        )
        print(f'[done] {emb}')

## Cell 6 — Train

In [ ]:
import importlib.util as _ilu

_spec = _ilu.spec_from_file_location(
    'train_type_b',
    Path(REPO_DIR) / 'src' / 'pipelines' / 'training' / 'type-b' / 'train_type_b.py',
)
_mod = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

total = len(MODELS) * len(EMBEDDINGS)
for i, model_name in enumerate(MODELS):
    for j, emb_name in enumerate(EMBEDDINGS):
        n = i * len(EMBEDDINGS) + j + 1
        print(f'\n[{n}/{total}] {model_name} × {emb_name}')
        _mod.run_experiment(
            model_name=model_name,
            embedding_name=emb_name,
            epochs=EPOCHS,
            device=device,
        )

print('\nAll done.')

## Cell 7 — Results & Loss Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics_dir = Path(REPO_DIR) / 'src' / 'pipelines' / 'results' / 'metrics'

# Summary table
summary_path = metrics_dir / 'results_summary.csv'
if summary_path.exists():
    df = pd.read_csv(summary_path)
    display(df.sort_values('top_1_acc', ascending=False))
else:
    print('No results_summary.csv yet.')

# Loss curves
for model_name in MODELS:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f'Loss Curves — {model_name} (Type-B)')
    plotted = False
    for emb_name in EMBEDDINGS:
        log_path = metrics_dir / f'b_{model_name}_{emb_name}_epoch_log.csv'
        if not log_path.exists():
            continue
        log = pd.read_csv(log_path)
        axes[0].plot(log['epoch'], log['train_loss'], label=emb_name)
        axes[1].plot(log['epoch'], log['val_loss'],   label=emb_name)
        plotted = True
    if plotted:
        for ax, title in zip(axes, ['Train Loss', 'Val Loss']):
            ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True, alpha=0.3)
        figures_dir = Path(REPO_DIR) / 'src' / 'pipelines' / 'results' / 'figures'
        figures_dir.mkdir(parents=True, exist_ok=True)
        out = figures_dir / f'loss_curves_{model_name}.png'
        fig.savefig(out, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved → {out}')

## Cell 8 — (Optional) Gemini Comparison

In [ ]:
if RUN_GEMINI:
    # os.environ['GEMINI_API_KEY'] = 'your-key-here'
    gemini_script = Path(REPO_DIR) / 'src' / 'pipelines' / 'evaluation' / 'gemini_comparison.py'
    subprocess.run([sys.executable, str(gemini_script)], cwd=REPO_DIR, check=True)
else:
    print('Skipping Gemini (RUN_GEMINI=False)')